In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import numpy as np
import re
import textwrap

# IMPORTS FOR WORDCLOUD
from wordcloud import WordCloud
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

# Ignore warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


# 1. CONFIGURATION AND LOADING

INPUT_FILE = ''
SIMILARITY_THRESHOLD = 0.4  # Threshold
SBERT_MODEL_NAME = 'paraphrase-multilingual-mpnet-base-v2'

try:
    df = pd.read_csv(INPUT_FILE)
    print(f"File '{INPUT_FILE}' uploaded successfully.")

    # Data cleaning
    df['comment_like_count'] = pd.to_numeric(df['comment_like_count'], errors='coerce').fillna(0).astype(int)
    df['no_names'] = df['no_names'].fillna('').astype(str)
    df['comment_message_clean'] = df['comment_message_clean'].fillna('').astype(str)

    # Identify top 10 parent comments based on number of replies
    replies = df[df['comment_parent'].notna()].copy()
    reply_counts = replies.groupby('comment_parent').size()
    top_10_parents_series = reply_counts.nlargest(10)
    top_10_parent_ids = top_10_parents_series.index.tolist()

    # Filter nodes for the graph
    top_10_parent_rows = df[df['comment_id'].isin(top_10_parent_ids)].copy()
    all_reply_rows = df[df['comment_parent'].isin(top_10_parent_ids)].copy()
    graph_df = pd.concat([top_10_parent_rows, all_reply_rows]).drop_duplicates(subset=['comment_id'])


    # 2. SIMILARITY CALCULATION

    print("Calculating semantic similarity")
    parent_text_map = top_10_parent_rows.set_index('comment_id')['no_names'].to_dict()
    parent_texts = [parent_text_map.get(pid, '') for pid in top_10_parent_ids]

    model = SentenceTransformer(SBERT_MODEL_NAME)
    embeddings = model.encode(parent_texts, convert_to_tensor=True, show_progress_bar=True)

    # 3. GRAPH CONSTRUCTION

    print("Building the graph")
    G = nx.Graph()

    # Add nodes
    for _, row in graph_df.iterrows():
        node_id = row['comment_id']
        likes = row['comment_like_count']
        G.add_node(node_id, likes=likes)
        G.nodes[node_id]['is_parent'] = (node_id in top_10_parent_ids)

    # Add reply edges
    for _, row in all_reply_rows.iterrows():
        if G.has_node(row['comment_parent']) and G.has_node(row['comment_id']):
            G.add_edge(row['comment_parent'], row['comment_id'], type='reply')

    # Add similarity edges
    similarity_edge_labels = {}
    for i in range(len(top_10_parent_ids)):
        for j in range(i + 1, len(top_10_parent_ids)):
            node_u = top_10_parent_ids[i]
            node_v = top_10_parent_ids[j]
            score = similarity_matrix[i, j]
            if score > SIMILARITY_THRESHOLD:
                G.add_edge(node_u, node_v, type='similarity', weight=score)
                similarity_edge_labels[(node_u, node_v)] = f"{score:.2f}"


    # 4. TEXT DATA AND MASK PREPARATION

    cluster_texts = {}
    for parent_id in top_10_parent_ids:
        p_text = df.loc[df['comment_id'] == parent_id, 'comment_message_clean'].values
        c_text_list = df.loc[df['comment_parent'] == parent_id, 'comment_message_clean'].values
        full_text = (" ".join(p_text) if len(p_text) > 0 else "") + " " + \
                    (" ".join(c_text_list) if len(c_text_list) > 0 else "")
        cluster_texts[parent_id] = full_text

    def create_circular_mask(h, w, radius_ratio=0.9):
        center_x, center_y = w // 2, h // 2
        Y, X = np.ogrid[:h, :w]
        dist_from_center = np.sqrt((X - center_x)**2 + (Y - center_y)**2)
        max_radius = min(center_x, center_y) * radius_ratio
        mask = dist_from_center > max_radius
        return mask.astype(int) * 255


    # 5. VISUALIZATION

    print("Starting visualization")
    plt.figure(figsize=(25, 25))
    ax = plt.gca()

    # Layout setup
    parent_subgraph = G.subgraph(top_10_parent_ids)
    pos_parents = nx.circular_layout(parent_subgraph)
    pos = nx.spring_layout(G, pos=pos_parents, fixed=top_10_parent_ids, k=0.15, iterations=100, seed=42)

    # Draw Reply Nodes (Blue)
    reply_nodes = [n for n in G.nodes() if not G.nodes[n].get('is_parent', False)]
    reply_sizes = [G.nodes[n]['likes'] * 3 + 40 for n in reply_nodes]

    nx.draw_networkx_nodes(G, pos, nodelist=reply_nodes, node_size=reply_sizes, node_color='#33CFFF', alpha=0.7)

    # Draw Edges
    reply_edges = [(u, v) for u, v, d in G.edges(data=True) if d['type'] == 'reply']
    similarity_edges = [(u, v) for u, v, d in G.edges(data=True) if d['type'] == 'similarity']
    similarity_weights = [G[u][v]['weight'] * 5 for u, v in similarity_edges]

    nx.draw_networkx_edges(G, pos, edgelist=reply_edges, edge_color='gray', alpha=0.3, arrows=False)

    # Solid lines for similarity
    nx.draw_networkx_edges(G, pos, edgelist=similarity_edges, edge_color='green',
                           width=similarity_weights, alpha=0.6, style='solid')

    nx.draw_networkx_edge_labels(G, pos, edge_labels=similarity_edge_labels, font_color='black', font_size=10)

    # Generate Word Clouds
    print("Generating Word Clouds")
    IMG_SIZE = 500
    mask = create_circular_mask(IMG_SIZE, IMG_SIZE, radius_ratio=0.85)
    max_likes = max([G.nodes[n]['likes'] for n in top_10_parent_ids]) if top_10_parent_ids else 1

    for i, parent_id in enumerate(top_10_parent_ids):
        text_content = cluster_texts.get(parent_id, "")
        if not text_content.strip(): continue

        wc = WordCloud(background_color="white", mode="RGB",
                       width=IMG_SIZE, height=IMG_SIZE, mask=mask,
                       colormap='viridis', max_words=40,
                       repeat=False, scale=1.5).generate(text_content)

        image_array = wc.to_array()
        full_mask = create_circular_mask(image_array.shape[0], image_array.shape[1], radius_ratio=1.0)
        alpha_channel = np.where(full_mask > 0, 0, 255).astype(np.uint8)
        final_image = np.dstack((image_array, alpha_channel))

        curr_likes = G.nodes[parent_id]['likes']
        zoom_factor = 0.12 + (curr_likes / max_likes) * 0.15

        im = OffsetImage(final_image, zoom=zoom_factor)
        ab = AnnotationBbox(im, pos[parent_id], frameon=False)
        ax.add_artist(ab)

        label_text = f"t{i+1}"
        offset_distance = zoom_factor * 1
        label_y = pos[parent_id][1] - offset_distance - 0.05

        ax.text(pos[parent_id][0], label_y,
                label_text,
                horizontalalignment='center',
                verticalalignment='top',
                fontsize=18,
                fontweight='bold',
                color='black',
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.7))

    #  SIDE LEGEND
    print("Creating Side Legend...")
    legend_content = "--- LEGEND ---\n\n"

    for i, parent_id in enumerate(top_10_parent_ids):
        raw_text = df.loc[df['comment_id'] == parent_id, 'no_names'].values[0]
        clean_text = str(raw_text).replace('\n', ' ').strip()
        # Remove Emoji / special characters
        clean_text = re.sub(r'[^\w\s,.\'àèéìòùÀÈÉÌÒÙ?!:;-]', '', clean_text)

        # Placeholder for empty comments
        if not clean_text:
            clean_text = "[This comment might contain a photo or a link]"

        # Wrap text
        wrapped_text = textwrap.fill(clean_text, width=60)
        legend_content += f"t{i+1}:\n{wrapped_text}\n\n"

    # External Legend placement
    plt.text(1.02, 1.0,
             legend_content,
             transform=ax.transAxes,
             fontsize=14,
             fontfamily='sans-serif',
             verticalalignment='top',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=1.0, edgecolor='gray'))

    plt.title(f'Network Analysis', fontsize=25)
    plt.axis('off')

    # Output filenames
    png_filename = 'comment_network_final_v2.png'
    pdf_filename = 'comment_network_final_1.pdf'

    # Save as PNG
    plt.savefig(png_filename, bbox_inches='tight', dpi=200)

    # Save as PDF (Vector format)
    plt.savefig(pdf_filename, format='pdf', bbox_inches='tight')

    plt.close()

    print(f"\nGraphs saved:")
    print(f"1. {png_filename} (Image preview)")
    print(f"2. {pdf_filename} (High quality PDF for Thesis)")

    from IPython.display import Image, display
    # Display the image
    display(Image(png_filename, width=1000))

except FileNotFoundError:
    print(f"\n Error: File not found.")
except Exception as e:
    print(f"\nUnexpected error: {e}")
    import traceback
    traceback.print_exc()

MATRIX

In [ ]:
import torch
# --- END OF NEW IMPORTS ---

# Ignore FutureWarning messages
warnings.simplefilter(action='ignore', category=FutureWarning)

# S-BERT configuration
SBERT_MODEL_NAME = 'paraphrase-multilingual-mpnet-base-v2'

try:
    # Load the CSV file
    df = pd.read_csv('')

    # --- 1. Identify the 10 comments with the most replies ---
    replies = df[df['comment_parent'].notna()].copy()
    reply_counts = replies.groupby('comment_parent').size()
    top_10_parents_series = reply_counts.nlargest(10)
    top_10_parent_ids = top_10_parents_series.index.tolist()

    # --- 2. Map labels (t1-t10) and get texts ---

    top_10_parent_rows = df[df['comment_id'].isin(top_10_parent_ids)]
    parent_text_map = top_10_parent_rows.set_index('comment_id')['no_names'].fillna('').to_dict()

    t_labels = [f"t{i+1}" for i in range(10)]
    parent_texts = [parent_text_map.get(pid, '') for pid in top_10_parent_ids]

    # --- 3. Print texts for each label ---
    print("--- Texts of the 10 Main Nodes ---")

    for i in range(len(t_labels)):
        label = t_labels[i]
        node_id = top_10_parent_ids[i]
        text = parent_texts[i]

        print(f"\n{label} (ID: {node_id}):")
        print(f"   {text}")

    # --- 4. Compute and print similarity matrix (Request 2) ---

    print("\nLoading S-BERT model")
    model = SentenceTransformer(SBERT_MODEL_NAME)

    print("Calculating embeddings (S-BERT)")
    # Compute dense vectors
    embeddings = model.encode(parent_texts, convert_to_tensor=True)

    print("Calculating similarity (Cosine Similarity on S-BERT)")
    # Compute similarity matrix (cosine)
    similarity_matrix = cosine_similarity(embeddings.cpu().numpy())

    print("\n--- Reciprocal Semantic Similarity Matrix (t1–t10) ---")

    # Create a pandas DataFrame for better readability
    similarity_df = pd.DataFrame(
        similarity_matrix,
        index=t_labels,
        columns=t_labels
    )

    # Printing settings for clear display
    pd.set_option('display.width', 1000)
    pd.set_option('display.precision', 3) # Show 3 decimal places

    print(similarity_df)

except FileNotFoundError:
    print(f"Error: The file was not found.")
except Exception as e:
    print(f"An unexpected error occurred:  {e}")
    import traceback
    traceback.print_exc()
